In [1]:
import os
import cv2
import joblib
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# --- 1. Jupyter-Safe Path Configuration ---
try:
    # This works if you export the notebook as a standard script later
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    print(" Running as a standard script configuration.")
except NameError:
    # This automatically grabs your current directory inside Jupyter Notebook
    BASE_DIR = os.getcwd()
    print(" Running interactively inside Jupyter Notebook.")

# Build absolute paths pointing exactly to your asset locations
MODEL_DIR = os.path.join(BASE_DIR, "model", "keypoint_classifier")
LANDMARKER_PATH = os.path.join(MODEL_DIR, "hand_landmarker.task")
CLASSIFIER_PATH = os.path.join(MODEL_DIR, "gesture_classifier.pkl")

# Define our human-readable text map matching our IDs
GESTURE_NAMES = {
    "0": "Open Palm",
    "1": "Closed Fist",
    "2": "Thumbs Up",
    "3": "Peace Sign",
    "4": "OK Sign"
}

# --- 2. Load Both Model Brains ---
if not os.path.exists(CLASSIFIER_PATH):
    raise FileNotFoundError(f" Missing trained classifier at: {CLASSIFIER_PATH}\n Please run your training script block first to generate this file!")

print(" Loading trained gesture classifier...")
model = joblib.load(CLASSIFIER_PATH)

print(" Initializing MediaPipe Hand Landmarker...")
base_options = python.BaseOptions(model_asset_path=LANDMARKER_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.7,
    min_hand_presence_confidence=0.5
)
landmarker = vision.HandLandmarker.create_from_options(options)

# --- 3. Normalization Helper ---
def normalize_landmarks(hand_landmarks_list):
    landmark_list = []
    for lm in hand_landmarks_list:
        landmark_list.append([lm.x, lm.y])
    np_landmarks = np.array(landmark_list)
    wrist = np_landmarks[0]
    normalized = np_landmarks - wrist
    flattened = normalized.flatten().tolist()
    max_val = max(max(abs(val) for val in flattened), 1e-6)
    return [val / max_val for val in flattened]

# --- 4. Live Video Prediction Loop ---
cap = cv2.VideoCapture(0)
print("\n Live Prediction Engine Active! Press 'q' while focused on the camera pop-up to exit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print(" Failed to grab frame from webcam.")
        break

    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    # Track coordinates using landmarker
    results = landmarker.detect(mp_image)

    if results.hand_landmarks:
        current_landmarks = results.hand_landmarks[0]
        
        # Draw green skeleton points on screen
        for lm in current_landmarks:
            cx, cy = int(lm.x * w), int(lm.y * h)
            cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)
            
        # Extract features and predict using our MLP model
        input_data = normalize_landmarks(current_landmarks)
        prediction = str(model.predict([input_data])[0]) # Returns string/int label like "3"
        
        # Map string ID to our human text name
        gesture_text = GESTURE_NAMES.get(prediction, f"Unknown ID ({prediction})")
        
        # Get wrist coordinate to position text cleanly above the hand
        wrist_x = int(current_landmarks[0].x * w)
        wrist_y = int(current_landmarks[0].y * h) - 30
        
        # Draw the predicted word on screen in blue
        cv2.putText(frame, gesture_text, (max(10, wrist_x - 50), max(30, wrist_y)), 
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 0), 2, cv2.LINE_AA)

    cv2.imshow('Real-Time Gesture Classifier', frame)
    
    # Check if 'q' key is pressed to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Inference closed smoothly.")

 Running interactively inside Jupyter Notebook.
 Loading trained gesture classifier...
 Initializing MediaPipe Hand Landmarker...

 Live Prediction Engine Active! Press 'q' while focused on the camera pop-up to exit.
Inference closed smoothly.
